# A.X-Encoder 문맥 적절성 분류기 — 단계별 실행

`train_ax_encoder.py`를 **import해서** 쓴다. 학습 로직·지표 정의를 이 노트북에 복제하지 않는다 —
복제하면 노트북 결과와 스크립트 결과가 조용히 달라진다.

## 실행 순서
1. 커널을 프로젝트 `venv`로 선택 (`venv/Scripts/python.exe`)
2. 셀 1~5 (세팅·데이터 확인) → 빠르다
3. 셀 6 (fold 1개만) → 설정이 맞는지 2~3분에 확인
4. 셀 7 (전체 CV) → RTX 3050 기준 10~13분, fold마다 표가 갱신된다
5. 셀 8~10 (지표·슬라이스·기저율)
6. 셀 11 (최종 모델 저장) → 3분
7. 셀 12 (추론 확인)

## VRAM 주의 (6GB 카드)
모델 하나가 fp32 가중치 0.6GB + AdamW 상태 1.2GB + 활성값을 쓴다. 커널이 살아있는 채로
모델을 새로 만들면 **이전 모델이 GPU에 남아 두 번째 실행에서 OOM**이 난다.
그래서 모든 학습 셀은 `del model` → `free_gpu()`로 끝난다. 이 두 줄을 지우지 말 것.

OOM이 나면: `BATCH = 8` → 그래도 나면 `BATCH, MAX_LEN = 8, 384`.

In [ ]:
# [1] 세팅 — 학습 로직은 스크립트에서 가져온다
import os, sys, json, gc, pathlib
import numpy as np, pandas as pd, torch


def find_root():
    """프로젝트 루트. 노트북을 어디서 열어도 찾게 한다 (cwd 자신 + 모든 상위 디렉터리)."""
    env = os.environ.get('CAREER_JIKIMI_ROOT')
    if env and (pathlib.Path(env) / 'training' / 'train_ax_encoder.py').exists():
        return pathlib.Path(env)
    cwd = pathlib.Path.cwd()
    for base in (cwd, *cwd.parents):
        if (base / 'training' / 'train_ax_encoder.py').exists():
            return base
    raise RuntimeError(f'프로젝트 루트를 못 찾음 (cwd={cwd}). '
                       '노트북을 프로젝트 안에서 열거나 환경변수 CAREER_JIKIMI_ROOT를 설정할 것.')


ROOT = find_root()
sys.path.insert(0, str(ROOT / 'training'))
os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')

from train_ax_encoder import (set_seed, PairDataset, collate, train_one, predict,
                              metrics_at, fresh_model, choose_threshold, slice_metrics,
                              MODEL_NAME)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = DEVICE == 'cuda'
print(f'ROOT   = {ROOT}')
print(f'device = {DEVICE}  amp = {USE_AMP}  model = {MODEL_NAME}')
if DEVICE == 'cuda':
    print(f'GPU    = {torch.cuda.get_device_name(0)}  '
          f'{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f}GB')
else:
    print('경고: CPU로는 5-fold CV가 몇 시간 걸린다. CUDA 설치를 먼저 확인할 것.')

In [ ]:
# [2] GPU 메모리 유틸 — 6GB 카드에서 반복 실행의 전제조건
def gpu_status(tag=''):
    if not torch.cuda.is_available():
        return
    tot = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'  {tag}GPU 사용 {torch.cuda.memory_allocated() / 1024**3:.2f}GB / '
          f'예약 {torch.cuda.memory_reserved() / 1024**3:.2f}GB / 전체 {tot:.1f}GB')


def free_gpu():
    """호출 전에 반드시 caller에서 `del model` 을 먼저 할 것 —
    이 함수는 caller의 변수를 지울 수 없고, 참조가 남아있으면 메모리도 안 돌아온다."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gpu_status('해제 후 ')


gpu_status('현재 ')

In [ ]:
# [3] 설정 — 조정은 여기서만
CSV      = ROOT / 'dataset' / 'training_dataset_v3_1000_simple.csv'
FULL_CSV = ROOT / 'dataset' / 'training_dataset_v3_1000.csv'   # 슬라이스 복원용(분석 컬럼 보유)
OUT      = ROOT / 'train_out_nb'                               # 스크립트의 train_out을 덮지 않는다

FOLDS, EPOCHS, LR, BATCH, MAX_LEN, SEED = 5, 4, 2e-5, 16, 512, 42
TARGET_PRECISION = 0.95

OUT.mkdir(exist_ok=True)
set_seed(SEED)
print(f'csv    = {CSV.name}')
print(f'out    = {OUT}')
print(f'folds={FOLDS} epochs={EPOCHS} lr={LR} batch={BATCH} max_len={MAX_LEN} seed={SEED}')

In [ ]:
# [4] 데이터 로드 + 구조 확인
df = pd.read_csv(CSV, encoding='utf-8-sig')
assert {'pair_id', 'history', 'response', 'label'} <= set(df.columns), '필수 컬럼 누락'
y_all = (df.label == '부적절').astype(int).values

print(f'{len(df)}행 / pair {df.pair_id.nunique()}개 / 라벨 {dict(df.label.value_counts())}')

# 짝 구조 — 같은 response에 history만 다른 쌍이 많을수록 '응답만 보고 맞추기'가 막힌다
g = df.groupby('pair_id')
print(f"같은 response + 다른 history : {int((g.response.nunique() == 1).sum())}쌍")
print(f"같은 history + 다른 response : {int((g.history.nunique() == 1).sum())}쌍")

# history 반복은 fold 간 문맥 누출로 이어진다 (셀 5에서 실제 누출량을 센다)
print(f'서로 다른 history {df.history.nunique()}개 — 반복 분포 '
      f'{dict(sorted(df.history.value_counts().value_counts().items()))}')
df.head(3)

In [ ]:
# [5] 분할 만들기 + fold 간 history 누출 측정
#
# 분할은 pair_id 그룹 단위 — 같은 response의 쌍둥이 행을 train/val에 가르지 않는다.
# 다만 dataset/README_dataset_v25.md 는 room_id 단위가 더 보수적이라고 권한다.
# v3 CSV에는 room_id가 없으므로, 대신 같은 history가 양쪽에 걸친 비율을 직접 센다.
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

skf = StratifiedGroupKFold(FOLDS, shuffle=True, random_state=SEED)
splits = list(skf.split(df, y_all, groups=df.pair_id))

hist = df.history.tolist()
leak = []
for k, (tr, va) in enumerate(splits):
    seen = {hist[i] for i in tr}
    n = sum(1 for i in va if hist[i] in seen)
    leak.append(dict(fold=k + 1, n_train=len(tr), n_val=len(va),
                     history누출=n, 비율=round(n / len(va), 3)))
leak_df = pd.DataFrame(leak)
print(f"전체 history 누출: {leak_df.history누출.sum()}/{len(df)} "
      f"= {leak_df.history누출.sum() / len(df):.1%}  <- 이만큼 OOF 점수가 낙관적이다")

tok = AutoTokenizer.from_pretrained(MODEL_NAME)


def make_loader(sub, shuffle):
    return DataLoader(PairDataset(sub, tok, MAX_LEN), batch_size=BATCH, shuffle=shuffle,
                      num_workers=0, collate_fn=lambda b: collate(b, tok))


leak_df

In [ ]:
# [6] fold 1개만 — 설정이 맞는지 먼저 확인하고 넘어간다 (2~3분)
#     여기서 OOM이 나면 셀 3의 BATCH를 8로 낮추고 커널을 재시작할 것
tr, va = splits[0]
model = fresh_model(MODEL_NAME, DEVICE)
gpu_status('모델 로드 후 ')

train_one(model, make_loader(df.iloc[tr], True), make_loader(df.iloc[va], False),
          DEVICE, EPOCHS, LR, USE_AMP)
gpu_status('학습 후 ')

del model
free_gpu()

In [ ]:
# [7] 전체 CV — fold가 끝날 때마다 표가 갱신된다 (10~13분)
from sklearn.metrics import roc_auc_score

oof = np.zeros(len(df))
rows = []
for k, (tr, va) in enumerate(splits):
    print(f'--- fold {k + 1}/{FOLDS} (train {len(tr)} / val {len(va)})')
    model = fresh_model(MODEL_NAME, DEVICE)
    train_one(model, make_loader(df.iloc[tr], True), make_loader(df.iloc[va], False),
              DEVICE, EPOCHS, LR, USE_AMP)
    oof[va], _ = predict(model, make_loader(df.iloc[va], False), DEVICE, USE_AMP)
    rows.append(dict(fold=k + 1, n_val=len(va),
                     auc=round(roc_auc_score(y_all[va], oof[va]), 4),
                     acc=round((( oof[va] >= .5).astype(int) == y_all[va]).mean(), 4)))
    display(pd.DataFrame(rows))
    del model
    free_gpu()

np.save(OUT / 'oof.npy', oof)
pd.DataFrame({'no': df.no if 'no' in df else range(len(df)),
              'pair_id': df.pair_id, 'label': df.label,
              'prob_inappropriate': oof.round(4)}
             ).to_csv(OUT / 'oof_predictions.csv', index=False, encoding='utf-8-sig')
print(f'\nOOF AUC = {roc_auc_score(y_all, oof):.4f}   (어휘 baseline 0.725를 크게 넘어야 성공)')

In [ ]:
# [8] threshold 선택 + 스윕
chosen = choose_threshold(y_all, oof, TARGET_PRECISION)
print(f'채택 threshold = {chosen}  (목표 precision {TARGET_PRECISION})')
display(pd.DataFrame([metrics_at(y_all, oof, chosen), metrics_at(y_all, oof, 0.5)],
                     index=['at_chosen', 'at_0.5']))

pd.DataFrame([metrics_at(y_all, oof, t) for t in np.arange(0.30, 0.996, 0.05)])

In [ ]:
# [9] 슬라이스 — difficulty=hard 가 무너지는지가 핵심 진단
#     simple CSV에는 분석 컬럼이 없으므로 전체 CSV를 `no`로 join해 복원한다
sl = slice_metrics(df, y_all, oof, chosen)
if not sl:
    full = pd.read_csv(FULL_CSV, encoding='utf-8-sig')
    m = df.merge(full[['no', 'pair_id', 'label', 'source', 'source_type', 'difficulty']],
                 on='no', suffixes=('', '_f'), validate='one_to_one')
    assert len(m) == len(df), 'join으로 행 수가 바뀌었다'
    # slice_metrics는 위치로 y_all/oof를 참조한다 — merge가 순서를 바꾸면 조용히 틀린 표가 나온다
    assert (m.no.values == df.no.values).all(), 'merge가 행 순서를 바꿨다'
    assert (m.pair_id == m.pair_id_f).all(), 'join이 다른 행을 붙였다 (pair_id 불일치)'
    assert (m.label == m.label_f).all(), 'join이 다른 행을 붙였다 (label 불일치)'
    sl = slice_metrics(m, y_all, oof, chosen)
    print(f'simple CSV -> {FULL_CSV.name} 을 join해 슬라이스 복원')

pd.DataFrame(sl).T.sort_values('n', ascending=False) if sl else '슬라이스 컬럼 없음'

In [ ]:
# [10] 실전 기저율 민감도 — 이 표가 이 노트북에서 가장 중요하다
#
# 데이터셋은 부적절 50%로 설계됐지만 실서비스의 오발송은 희귀하다.
# precision은 기저율에 정면으로 좌우되므로, 위의 0.95는 실전 수치가 아니다.
# requirements.md: "flag rate ... 이 값이 높으면 precision과 무관하게 기능을 못 쓴다"
tpr = (oof[y_all == 1] >= chosen).mean()   # recall
fpr = (oof[y_all == 0] >= chosen).mean()   # 정상 메시지를 막는 비율
print(f'threshold {chosen}: recall = {tpr:.3f}, FPR = {fpr:.4f}')

rows = []
for base in (0.50, 0.10, 0.05, 0.02, 0.01, 0.005, 0.001):
    fr = base * tpr + (1 - base) * fpr          # 전체 메시지 중 팝업이 뜨는 비율
    rows.append({'실전_오발송비율': f'{base:.1%}',
                 'precision': round(base * tpr / fr, 4),
                 '팝업률': round(fr, 4),
                 '만건당_팝업': int(fr * 10000),
                 '그중_진짜': int(fr * 10000 * base * tpr / fr)})
pd.DataFrame(rows)

In [ ]:
# [11] 최종 모델 — 전체 데이터로 학습해 저장 (3분)
model = fresh_model(MODEL_NAME, DEVICE)
train_one(model, make_loader(df, True), None, DEVICE, EPOCHS, LR, USE_AMP)

final_dir = OUT / 'final_model'
model.save_pretrained(final_dir)
tok.save_pretrained(final_dir)
(final_dir / 'threshold.json').write_text(
    json.dumps({'threshold': chosen, 'target_precision': TARGET_PRECISION,
                'label_map': {'0': '적절', '1': '부적절'}}, ensure_ascii=False, indent=2),
    encoding='utf-8')

# 스크립트의 report.json과 같은 모양으로 남긴다
report = {'model': MODEL_NAME, 'rows': len(df), 'folds': FOLDS, 'epochs': EPOCHS,
          'lr': LR, 'max_len': MAX_LEN, 'csv': CSV.name,
          'oof_auc': round(roc_auc_score(y_all, oof), 4),
          'chosen_threshold': chosen,
          'at_chosen': metrics_at(y_all, oof, chosen),
          'at_0.5': metrics_at(y_all, oof, 0.5),
          'slices': sl,
          'history_leak_rate': round(leak_df.history누출.sum() / len(df), 4)}
(OUT / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2),
                                 encoding='utf-8')
print(f'저장 완료: {final_dir}  (threshold={chosen})')

del model
free_gpu()

In [ ]:
# [12] 추론 확인 — FastAPI에 넣을 코드와 같은 클래스를 쓴다
from predict_example import ContextChecker

ck = ContextChecker(str(OUT / 'final_model'))
hist_work = ['팀장: 내일 배포 일정 확인해주세요.',
             '나: 네, 오전 중으로 정리하겠습니다.',
             '팀장: 롤백 계획도 포함해주세요.']
tests = [
    (hist_work, '넵, 롤백 절차까지 포함해서 문서로 공유드리겠습니다.'),   # 적절해야 한다
    (hist_work, '오늘 저녁 치킨 ㄱ? 양념 반 후라이드 반 어때 ㅋㅋ'),      # 부적절해야 한다
    (hist_work, '넵 확인했습니다.'),                                      # 일반 맞장구 — 경계 사례
]
for h, c in tests:
    v, p = ck.check(h, c)
    print(f'[{v:4} p={p:<8}] {c}')

del ck
free_gpu()

## 직접 문장을 넣어보려면

셀 12의 `ck`를 살려둔 채(마지막 두 줄을 지우고) 아래처럼 호출한다.

```python
ck.check(['친구A: 주말에 등산 갈래?', '나: 좋아 어디로?'], '내일 회의 자료 검토 부탁드립니다')
```

## 알려진 한계 (셀 5·10의 숫자가 근거)

- **OOF AUC는 낙관적이다** — 검증 행의 상당수가 학습에서 본 history다(셀 5). 실전은 모든 방이 처음 보는 방이다.
- **precision 0.95는 50/50 분포의 값이다** — 실전 기저율에서는 셀 10의 표를 봐야 한다.
- 앱의 `AI_VERDICT_THRESHOLD` 기본값과 여기서 고른 threshold가 다르면 조용히 다른 운영점에서 돌아간다.
  `final_model/threshold.json`의 값을 앱 설정으로 옮길 것.